# Física BR · Taller: Carga Eléctrica, Ley de Coulomb y Campo Eléctrico
### Física BR — Física de Electricidad y Magnetismo (Fundamentos)
**Profesor:** Juan David Betancur Ríos · **Semestre:** 2026-2

---
*Física interactiva para ingeniería.* La carga eléctrica, la fuerza de Coulomb y el campo eléctrico
de cargas puntuales son la base de toda la electricidad. Este taller abre el curso. Cada punto parte
de una situación real e incluye esquema, sliders, botón **"Mostrar solución"** y cuestionario.

**Convenciones:** tres cifras significativas · coma decimal · k = 8,99×10⁹ N·m²/C² · e = 1,60×10⁻¹⁹ C.

**Ecuaciones clave:**
- Carga cuantizada:  Q = n·e    (múltiplo entero de la carga elemental)
- Ley de Coulomb:  F = k·|q₁·q₂|/r²    (atractiva o repulsiva según los signos)
- Superposición:  la fuerza (o el campo) neto es la suma vectorial de las contribuciones
- Campo de carga puntual:  E = k·|Q|/r²    ·    Fuerza sobre una carga:  F = q·E

> **Uso:** `Entorno de ejecución → Ejecutar todo`. Competencias: [CL, UFT, UCM, PRC, ULC].

---
> © 2026 **Juan David Betancur Ríos** (docente de física para ingeniería) y **Nora Yamile Rojas Cataño** (aporte pedagógico y metodología de evaluación). Material educativo de uso académico (Física BR). Todos los derechos reservados.
> *Elaborado con apoyo de herramientas de inteligencia artificial, bajo la supervisión y criterio pedagógico de los autores.*

---
## Punto 1 — Cuantización de la carga (Ing. Electrónica/Materiales)  ★★  [CL, UCM]
Un cuerpo adquiere un exceso de  5,00×10¹²  electrones. Se busca su carga neta, y cuántos electrones
equivalen a 1,00 nC.

**Carga cuantizada:**  Q = n·e    (e = 1,60×10⁻¹⁹ C es la carga elemental)

> 🔧 **Aplicación en ingeniería:** la carga siempre viene en múltiplos de la carga del electrón. En
> electrónica y nanotecnología, controlar cantidades de carga (en sensores, memorias, dispositivos)
> exige entender esta cuantización. La corriente misma es carga en movimiento.

> 🤖 **TIP-IA — Traductor de representaciones:**
> Pregúntale a un asistente: *"Explícame qué significa físicamente que la carga esté 'cuantizada' y por qué no puede existir media carga del electrón. ¿Qué analogía cotidiana captura la idea de cuantización? No me des la definición de memoria, dame la intuición."* 

In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

# Registro global para la calificación formativa ponderada
try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))


def _num_coma(s):
    import re as _re
    # reemplaza punto decimal por coma en números dentro de un string ya formateado
    return _re.sub(r"(\d)\.(\d)", r"\1,\2", s)
def print_es(*args, **kwargs):
    args2=[_num_coma(a) if isinstance(a,str) else a for a in args]
    print(*args2, **kwargs)

def mostrar_solucion(func_solucion):
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output(); func_solucion()
    boton.on_click(_click)
    display(boton, salida)

try:
    print_es
except NameError:
    def print_es(*args, **kwargs):
        import re as _re
        print(*[_re.sub(r"(\d)\.(\d)", r"\1,\2", a) if isinstance(a, str) else a for a in args], **kwargs)
# --- Cuestionarios Física BR v3: opciones barajadas, crédito por intento, nota sobre el total ---
import random as _rnd_br
try:
    PESOS_BR
except NameError:
    PESOS_BR = {}      # id_preg -> peso de cada pregunta mostrada
try:
    INTENTOS_BR
except NameError:
    INTENTOS_BR = {}   # id_preg -> [intentos, resuelta]
_CREDITO_BR = {1: 1.0, 2: 0.5, 3: 0.25}   # crédito según el intento en que se acierta
_MAX_INT_BR = 3

def _evaluar_intento(id_preg, ok, peso, explicacion, msg_ok, msg_mal):
    est = INTENTOS_BR.setdefault(id_preg, [0, False])
    if est[1]:
        print("✔ Ya resolviste esta pregunta; tu crédito quedó registrado.")
        if explicacion: print_es("→ " + explicacion)
        return
    if est[0] >= _MAX_INT_BR:
        print("⛔ Agotaste los 3 intentos con crédito: esta pregunta ya no suma.")
        if explicacion: print_es("→ " + explicacion)
        return
    est[0] += 1
    n = est[0]
    if ok:
        est[1] = True
        cred = _CREDITO_BR.get(n, 0.0)
        REGISTRO_BR[id_preg] = (cred, float(peso))
        print_es(msg_ok + f"  (intento {n}: crédito {int(cred*100)} %)")
        if explicacion: print_es("→ " + explicacion)
    else:
        REGISTRO_BR.setdefault(id_preg, (0.0, float(peso)))
        if n < _MAX_INT_BR:
            print(msg_mal + f"  Intento {n} de {_MAX_INT_BR}. Revisa tu razonamiento antes de volver a intentar.")
        else:
            print(msg_mal + " Agotaste los 3 intentos: esta pregunta ya no suma. Estudia la explicación:")
            if explicacion: print_es("→ " + explicacion)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    orden = list(range(len(opciones))); _rnd_br.shuffle(orden)   # orden distinto en cada sesión
    correcta = orden.index(indice_correcto)
    letras = "abcdefgh"
    etiquetas = [f"{letras[i]}) {opciones[j]}" for i, j in enumerate(orden)]
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=etiquetas, value=None, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            if radio.index is None:
                print("⚠ Elige una opción."); return
            _evaluar_intento(id_preg, radio.index == correcta, peso, explicacion,
                             "✅ ¡Correcto!", "❌ Incorrecto.")
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.strip().replace(",", "."))
            except ValueError: print("⚠ Escribe un número (usa coma o punto decimal)."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            _evaluar_intento(id_preg, ok, peso, explicacion,
                             f"✅ ¡Correcto!  ({valor_correcto:.3g} {unidad})", "❌ Fuera de rango.")
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not PESOS_BR:
                print("No se encontraron cuestionarios en este taller."); return
            peso_total = sum(PESOS_BR.values())
            peso_ok = sum(float(c)*p for c, p in REGISTRO_BR.values())
            nota = 5.0*peso_ok/peso_total if peso_total > 0 else 0.0
            total = len(PESOS_BR)
            resp = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0])[0] > 0)
            prim = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0, False])[1] and INTENTOS_BR[k][0] == 1)
            print("="*46)
            print("          CALIFICACIÓN FORMATIVA")
            print("="*46)
            print(f"  Preguntas respondidas:       {resp} de {total}")
            print(f"  Acertadas al primer intento: {prim}")
            print_es(f"  Puntaje ponderado:           {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*46)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*46)
            if resp < total: print(f"  ⚠ Faltan {total-resp} preguntas: cuentan como 0.")
            if nota >= 3.5: print("  ✅ ¡Buen trabajo! Dominas los conceptos.")
            elif nota >= 3.0: print("  🟡 Vas bien, repasa lo que fallaste.")
            else: print("  🔴 Repasa la teoría y vuelve a los puntos.")
            print("\n  Crédito por pregunta: 100 % al primer intento,")
            print("  50 % al segundo, 25 % al tercero. Predice y")
            print("  razona ANTES de verificar: adivinar cuesta nota.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>📊 Tu calificación del taller</h3>"
        "<i>La nota se calcula sobre TODAS las preguntas del taller y premia acertar al primer intento.</i>"),
        boton, salida]))


k = 8.99e9
e = 1.60e-19

def esquema():
    fig,ax=plt.subplots(figsize=(7.5,3.5))
    # la carga viene en "paquetes" indivisibles de e
    for i in range(5):
        ax.scatter([i*1.2+0.5],[1],s=400,color='#c0392b',zorder=5,edgecolor='white',linewidth=1.5)
        ax.text(i*1.2+0.5,1,'e',ha='center',va='center',color='white',fontsize=11,fontweight='bold')
    ax.text(3.2,2.2,'La carga siempre es un múltiplo entero de e',ha='center',fontsize=10,color='#2c3e50',fontweight='bold')
    ax.text(3.2,0.1,'Q = n·e   (n entero)   ·   e = 1,60×10⁻¹⁹ C',ha='center',fontsize=10,color='#8e44ad',fontweight='bold')
    ax.annotate('',xy=(0.5,1.6),xytext=(5.3,1.6),arrowprops=dict(arrowstyle='<->',color='#7f8c8d',lw=1))
    ax.text(2.9,1.75,'5 electrones = 5e',ha='center',fontsize=8,color='#7f8c8d')
    ax.text(3.2,-0.6,'No existe media carga: la carga está CUANTIZADA',ha='center',fontsize=9,color='#c0392b')
    ax.set_xlim(-0.3,6.5); ax.set_ylim(-1,2.6); ax.axis('off')
    ax.set_title('Cuantización de la carga eléctrica')
    plt.tight_layout(); plt.show()
esquema()

def carga(n_exp=12.0):
    n=5*10**n_exp; Q=n*e
    ns=np.logspace(9,15,100); Qs=ns*e
    plt.figure(figsize=(9,4)); plt.loglog(ns,Qs,lw=2,color='navy')
    plt.scatter([n],[Q],color='red',s=60,zorder=5,label=f'n={n:.1e} → Q={Q:.2e} C')
    plt.xlabel('número de electrones n'); plt.ylabel('carga |Q| (C)')
    plt.title('Carga vs número de electrones (Q=n·e)'); plt.legend(); plt.grid(alpha=.3,which='both')
    plt.tight_layout(); plt.show()

interact(carga, n_exp=FloatSlider(value=12.0,min=9,max=15,step=0.5,description='exp (5×10^n)'));

def solucion():
    import re as _re
    def _pc(*a,**k):
        def _f(t):
            if not isinstance(t,str): return t
            _sup='⁰¹²³⁴⁵⁶⁷⁸⁹'
            def _sci(m):
                mant=m.group(1); exp=int(m.group(2))
                e=('⁻' if exp<0 else '')+''.join(_sup[int(d)] for d in str(abs(exp)))
                return mant+'×10'+e
            t=_re.sub(r'(\d+(?:\.\d+)?)[eE]([+-]?\d+)', _sci, t)
            _u=r'(?:s|km/s|km|m/s²|m/s|m|N·m|N|J|W|A|V|T|Ω|C|F|Hz|rad/s²|rad/s|rad|kg·m²/s|kg·m²|kg|Pa|kPa|µF|nC|µC|pF|mWb|Wb|µT|mT|kW|MW|mA|kΩ|MΩ|rpm|°)'
            def _c3(m):
                num=m.group(2); n=int(num)
                if n==0: return m.group(0)
                if len(num)==1: nv=num+'.00'
                elif len(num)==2: nv=num+'.0'
                else: nv=num
                return m.group(1)+nv+m.group(3)
            t=_re.sub(r'(?<![\d.,])([\s=(×])(\d{1,3})(\s'+_u+r'\b)', _c3, t)
            t=_re.sub(r'(\d)\.(\d)', r'\1,\2', t)
            return t
        print(*[_f(x) for x in a], **k)
    n=5e12; Q=n*e
    _pc(f"Carga neta: Q = n·e = 5,00×10¹² × 1,60×10⁻¹⁹ = {Q:.3g} C")
    _pc(f"(negativa, por ser exceso de electrones)")
    _pc(f"Electrones en 1,00 nC: N = Q/e = 1,00×10⁻⁹/1,60×10⁻¹⁹ = {1e-9/e:.3g}")
mostrar_solucion(solucion)

print("\n--- Cuestionario Punto 1 ---")
quiz_numerico("La carga neta de un cuerpo con 5,00×10¹² electrones en exceso (en µC) es:",0.8,0.04,"µC","Q=n·e=8,00×10⁻⁷ C=0,800 µC.", id_preg="Coulom_p1_1", peso=2.0)
quiz_opcion_multiple("Que la carga esté 'cuantizada' significa que:",
    ['Puede tomar cualquier valor real, como cualquier magnitud continua', 'Siempre es un múltiplo entero de la carga elemental e', 'Solo existe en metales, donde hay electrones libres', 'No se conserva: puede crearse o destruirse en cada choque'],1,
    "Q=n·e: la carga viene en múltiplos enteros de la carga elemental.", id_preg="Coulom_p1_2", peso=2.0)

# --- Estilo visual Física BR (apariencia global de los esquemas) ---
try:
    import matplotlib.pyplot as _plt_style
    _plt_style.rcParams['axes.spines.top']=False
    _plt_style.rcParams['axes.spines.right']=False
    _plt_style.rcParams['axes.titlesize']=12
    _plt_style.rcParams['axes.titleweight']='bold'
    _plt_style.rcParams['axes.titlecolor']='#2c3e50'
    _plt_style.rcParams['axes.labelsize']=10
    _plt_style.rcParams['axes.labelcolor']='#34495e'
    _plt_style.rcParams['font.size']=10
    _plt_style.rcParams['axes.grid']=True
    _plt_style.rcParams['grid.alpha']=0.25
    _plt_style.rcParams['grid.linestyle']='--'
    _plt_style.rcParams['axes.edgecolor']='#7f8c8d'
    _plt_style.rcParams['xtick.color']='#34495e'
    _plt_style.rcParams['ytick.color']='#34495e'
    _plt_style.rcParams['figure.facecolor']='white'
    _plt_style.rcParams['axes.facecolor']='#fcfcfd'
except Exception:
    pass
# --- fin estilo ---
quiz_opcion_multiple('DEFIENDE TU CONCLUSIÓN: un estudiante dice que un cuerpo puede tener una carga de 1,5 veces la del electrón. ¿Correcto o solo convincente?',
    ['Solo convincente: la carga está cuantizada; la carga neta de un cuerpo es siempre n·e con n entero',
     'Correcto: la carga eléctrica es una magnitud continua y puede tomar cualquier valor',
     'Correcto: si el cuerpo pierde un electrón y medio, su carga queda en 1,5 veces la carga del electrón',
     'Solo convincente: un cuerpo solo puede tener cargas pares de e: 2e, 4e, 6e…'],
    0,
    "No existe 'medio electrón' que transferir. Cualquier entero sirve (1e, 2e, 3e…), no solo los pares. Los quarks tienen carga fraccionaria, pero nunca aparecen aislados en un cuerpo.",
    id_preg='Coul_p1_concep_1', peso=2.0)

---
## Punto 2 — Fuerza de Coulomb entre dos cargas (Ing. Electrónica/Física)  ★★  [CL, UFT]
Dos cargas puntuales  q₁ = 3,00 µC  y  q₂ = 5,00 µC  están separadas  r = 0,200 m. Se busca la
fuerza eléctrica entre ellas.

**Ley de Coulomb:**  F = k·|q₁·q₂|/r²

> 🔧 **Aplicación en ingeniería:** la fuerza de Coulomb gobierna la atracción y repulsión entre
> cargas, base del funcionamiento de sensores capacitivos, precipitadores electrostáticos, impresión
> por tóner y micromanipulación. Su dependencia con 1/r² es idéntica en forma a la gravitación.

> 🤖 **TIP-IA — Pon a prueba tu intuición:**
> Pídele: *"Voy a sostener que si duplico la distancia entre dos cargas, la fuerza se reduce a la mitad. Dame el mejor contraargumento."* (Pista: la ley de Coulomb va con 1/r²). La IA como espejo de tu razonamiento, no como sustituto.

In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

# Registro global para la calificación formativa ponderada
try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))


def _num_coma(s):
    import re as _re
    # reemplaza punto decimal por coma en números dentro de un string ya formateado
    return _re.sub(r"(\d)\.(\d)", r"\1,\2", s)
def print_es(*args, **kwargs):
    args2=[_num_coma(a) if isinstance(a,str) else a for a in args]
    print(*args2, **kwargs)

def mostrar_solucion(func_solucion):
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output(); func_solucion()
    boton.on_click(_click)
    display(boton, salida)

try:
    print_es
except NameError:
    def print_es(*args, **kwargs):
        import re as _re
        print(*[_re.sub(r"(\d)\.(\d)", r"\1,\2", a) if isinstance(a, str) else a for a in args], **kwargs)
# --- Cuestionarios Física BR v3: opciones barajadas, crédito por intento, nota sobre el total ---
import random as _rnd_br
try:
    PESOS_BR
except NameError:
    PESOS_BR = {}      # id_preg -> peso de cada pregunta mostrada
try:
    INTENTOS_BR
except NameError:
    INTENTOS_BR = {}   # id_preg -> [intentos, resuelta]
_CREDITO_BR = {1: 1.0, 2: 0.5, 3: 0.25}   # crédito según el intento en que se acierta
_MAX_INT_BR = 3

def _evaluar_intento(id_preg, ok, peso, explicacion, msg_ok, msg_mal):
    est = INTENTOS_BR.setdefault(id_preg, [0, False])
    if est[1]:
        print("✔ Ya resolviste esta pregunta; tu crédito quedó registrado.")
        if explicacion: print_es("→ " + explicacion)
        return
    if est[0] >= _MAX_INT_BR:
        print("⛔ Agotaste los 3 intentos con crédito: esta pregunta ya no suma.")
        if explicacion: print_es("→ " + explicacion)
        return
    est[0] += 1
    n = est[0]
    if ok:
        est[1] = True
        cred = _CREDITO_BR.get(n, 0.0)
        REGISTRO_BR[id_preg] = (cred, float(peso))
        print_es(msg_ok + f"  (intento {n}: crédito {int(cred*100)} %)")
        if explicacion: print_es("→ " + explicacion)
    else:
        REGISTRO_BR.setdefault(id_preg, (0.0, float(peso)))
        if n < _MAX_INT_BR:
            print(msg_mal + f"  Intento {n} de {_MAX_INT_BR}. Revisa tu razonamiento antes de volver a intentar.")
        else:
            print(msg_mal + " Agotaste los 3 intentos: esta pregunta ya no suma. Estudia la explicación:")
            if explicacion: print_es("→ " + explicacion)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    orden = list(range(len(opciones))); _rnd_br.shuffle(orden)   # orden distinto en cada sesión
    correcta = orden.index(indice_correcto)
    letras = "abcdefgh"
    etiquetas = [f"{letras[i]}) {opciones[j]}" for i, j in enumerate(orden)]
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=etiquetas, value=None, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            if radio.index is None:
                print("⚠ Elige una opción."); return
            _evaluar_intento(id_preg, radio.index == correcta, peso, explicacion,
                             "✅ ¡Correcto!", "❌ Incorrecto.")
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.strip().replace(",", "."))
            except ValueError: print("⚠ Escribe un número (usa coma o punto decimal)."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            _evaluar_intento(id_preg, ok, peso, explicacion,
                             f"✅ ¡Correcto!  ({valor_correcto:.3g} {unidad})", "❌ Fuera de rango.")
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not PESOS_BR:
                print("No se encontraron cuestionarios en este taller."); return
            peso_total = sum(PESOS_BR.values())
            peso_ok = sum(float(c)*p for c, p in REGISTRO_BR.values())
            nota = 5.0*peso_ok/peso_total if peso_total > 0 else 0.0
            total = len(PESOS_BR)
            resp = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0])[0] > 0)
            prim = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0, False])[1] and INTENTOS_BR[k][0] == 1)
            print("="*46)
            print("          CALIFICACIÓN FORMATIVA")
            print("="*46)
            print(f"  Preguntas respondidas:       {resp} de {total}")
            print(f"  Acertadas al primer intento: {prim}")
            print_es(f"  Puntaje ponderado:           {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*46)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*46)
            if resp < total: print(f"  ⚠ Faltan {total-resp} preguntas: cuentan como 0.")
            if nota >= 3.5: print("  ✅ ¡Buen trabajo! Dominas los conceptos.")
            elif nota >= 3.0: print("  🟡 Vas bien, repasa lo que fallaste.")
            else: print("  🔴 Repasa la teoría y vuelve a los puntos.")
            print("\n  Crédito por pregunta: 100 % al primer intento,")
            print("  50 % al segundo, 25 % al tercero. Predice y")
            print("  razona ANTES de verificar: adivinar cuesta nota.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>📊 Tu calificación del taller</h3>"
        "<i>La nota se calcula sobre TODAS las preguntas del taller y premia acertar al primer intento.</i>"),
        boton, salida]))


k = 8.99e9
e = 1.60e-19

def esquema():
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(9,3.8))
    # IZQUIERDA: repulsión (cargas iguales) y atracción (opuestas)
    ax1.scatter([1],[2],s=350,color='#c0392b',zorder=5,edgecolor='white',linewidth=1.5)
    ax1.text(1,2,'+',ha='center',va='center',color='white',fontsize=16,fontweight='bold')
    ax1.scatter([3],[2],s=350,color='#c0392b',zorder=5,edgecolor='white',linewidth=1.5)
    ax1.text(3,2,'+',ha='center',va='center',color='white',fontsize=16,fontweight='bold')
    ax1.annotate('',xy=(0.2,2),xytext=(0.9,2),arrowprops=dict(arrowstyle='-|>',color='#e74c3c',lw=2.5))
    ax1.annotate('',xy=(3.8,2),xytext=(3.1,2),arrowprops=dict(arrowstyle='-|>',color='#e74c3c',lw=2.5))
    ax1.text(2,2.5,'REPULSIÓN\n(cargas iguales)',ha='center',fontsize=9,color='#c0392b',fontweight='bold')
    # atracción
    ax1.scatter([1],[0.5],s=350,color='#c0392b',zorder=5,edgecolor='white',linewidth=1.5)
    ax1.text(1,0.5,'+',ha='center',va='center',color='white',fontsize=16,fontweight='bold')
    ax1.scatter([3],[0.5],s=350,color='#2980b9',zorder=5,edgecolor='white',linewidth=1.5)
    ax1.text(3,0.5,'−',ha='center',va='center',color='white',fontsize=16,fontweight='bold')
    ax1.annotate('',xy=(1.8,0.5),xytext=(1.1,0.5),arrowprops=dict(arrowstyle='-|>',color='#27ae60',lw=2.5))
    ax1.annotate('',xy=(2.2,0.5),xytext=(2.9,0.5),arrowprops=dict(arrowstyle='-|>',color='#27ae60',lw=2.5))
    ax1.text(2,-0.3,'ATRACCIÓN\n(cargas opuestas)',ha='center',fontsize=9,color='#27ae60',fontweight='bold')
    ax1.set_xlim(-0.3,4.3); ax1.set_ylim(-0.9,3.1); ax1.axis('off')
    ax1.set_title('Repulsión y atracción',fontsize=10)
    # DERECHA: F vs r (1/r²)
    r=np.linspace(0.5,4,100); F=1/r**2
    ax2.plot(r,F,color='#8e44ad',lw=2.5)
    ax2.fill_between(r,0,F,color='#8e44ad',alpha=0.15)
    ax2.set_xlabel('distancia r'); ax2.set_ylabel('fuerza F')
    ax2.set_title('F ∝ 1/r² (cae rápido con la distancia)',fontsize=10)
    ax2.annotate('al doblar r,\nla fuerza baja a 1/4',xy=(2,0.25),xytext=(2.3,1.5),fontsize=8,color='#2c3e50',
                 arrowprops=dict(arrowstyle='->',color='#7f8c8d'))
    ax2.grid(alpha=0.2,linestyle='--')
    plt.suptitle('Ley de Coulomb: fuerza entre cargas',fontsize=11,fontweight='bold',color='#2c3e50')
    plt.tight_layout(); plt.show()
esquema()

def coulomb(q1_uC=3.0, q2_uC=5.0, r=0.20):
    q1=q1_uC*1e-6; q2=q2_uC*1e-6; F=k*q1*q2/r**2
    rs=np.linspace(0.05,0.6,100); Fs=k*q1*q2/rs**2
    plt.figure(figsize=(9,4)); plt.plot(rs,Fs,lw=2,color='crimson')
    plt.scatter([r],[F],color='blue',s=60,zorder=5,label=f'r={r:.2f} m → F={F:.2f} N')
    plt.xlabel('distancia r (m)'); plt.ylabel('fuerza F (N)')
    plt.title('Fuerza de Coulomb vs distancia (F ∝ 1/r²)'); plt.legend(); plt.grid(alpha=.3)
    plt.tight_layout(); plt.show()

interact(coulomb,
    q1_uC=FloatSlider(value=3.0,min=1,max=10,step=0.5,description='q1 (µC)'),
    q2_uC=FloatSlider(value=5.0,min=1,max=10,step=0.5,description='q2 (µC)'),
    r=FloatSlider(value=0.20,min=0.05,max=0.6,step=0.05,description='r (m)'));

def solucion():
    import re as _re
    def _pc(*a,**k):
        def _f(t):
            if not isinstance(t,str): return t
            _sup='⁰¹²³⁴⁵⁶⁷⁸⁹'
            def _sci(m):
                mant=m.group(1); exp=int(m.group(2))
                e=('⁻' if exp<0 else '')+''.join(_sup[int(d)] for d in str(abs(exp)))
                return mant+'×10'+e
            t=_re.sub(r'(\d+(?:\.\d+)?)[eE]([+-]?\d+)', _sci, t)
            _u=r'(?:s|km/s|km|m/s²|m/s|m|N·m|N|J|W|A|V|T|Ω|C|F|Hz|rad/s²|rad/s|rad|kg·m²/s|kg·m²|kg|Pa|kPa|µF|nC|µC|pF|mWb|Wb|µT|mT|kW|MW|mA|kΩ|MΩ|rpm|°)'
            def _c3(m):
                num=m.group(2); n=int(num)
                if n==0: return m.group(0)
                if len(num)==1: nv=num+'.00'
                elif len(num)==2: nv=num+'.0'
                else: nv=num
                return m.group(1)+nv+m.group(3)
            t=_re.sub(r'(?<![\d.,])([\s=(×])(\d{1,3})(\s'+_u+r'\b)', _c3, t)
            t=_re.sub(r'(\d)\.(\d)', r'\1,\2', t)
            return t
        print(*[_f(x) for x in a], **k)
    q1=3e-6; q2=5e-6; r=0.20; F=k*q1*q2/r**2
    _pc(f"F = k·|q1·q2|/r² = 8,99×10⁹ × (3,00×10⁻⁶ × 5,00×10⁻⁶)/(0,200)²")
    _pc(f"  = {F:.3g} N (repulsiva, por ser ambas positivas)")
    _pc("Si una carga fuera negativa, la fuerza sería atractiva (mismo valor).")
mostrar_solucion(solucion)

print("\n--- Cuestionario Punto 2 ---")
quiz_numerico("La fuerza entre q₁=3,00 µC y q₂=5,00 µC separadas 0,200 m es:",3.37,0.04,"N","F=k·q1·q2/r²≈3,37 N.", id_preg="Coulom_p2_3", peso=2.0)
quiz_opcion_multiple("Si se duplica la distancia entre las dos cargas, la fuerza de Coulomb:",
    ['Se duplica (F ∝ r)', 'Se reduce a la cuarta parte (F ∝ 1/r²)', 'No cambia (F depende de q)', 'Se reduce a la mitad (F ∝ 1/r)'],1,
    "F∝1/r²: al doblar r, la fuerza cae a 1/4.", id_preg="Coulom_p2_4", peso=2.0)
quiz_opcion_multiple("Entre dos cargas del mismo signo, la fuerza es:",
    ["Atractiva","Repulsiva","Nula","Depende de la distancia"],1,
    "Cargas de igual signo se repelen; de signo opuesto se atraen.", id_preg="Coulom_p2_5", peso=2.0)
quiz_opcion_multiple("DEFIENDE TU CONCLUSIÓN: alguien afirma: 'al triplicar la distancia entre dos cargas, la fuerza se reduce a la novena parte'. ¿Correcto o solo convincente?",
    ['Correcto: F = k·q₁·q₂/r²; con 3r la fuerza queda dividida por 3² = 9',
     'Correcto: se reduce a la novena parte porque las cargas también disminuyen al alejarse',
     'Solo convincente: F es inversamente proporcional a r, así que se reduce a un tercio',
     'Solo convincente: la fuerza eléctrica depende solo de las cargas, no de su distancia'],
    0,
    'Esta vez es CORRECTO. Las cargas no cambian al alejarse: todo el efecto viene del r² del denominador. La opción «se reduce a la novena parte…» acierta el resultado con una razón falsa.',
    id_preg='Coul_p2_concep_2', peso=2.0)

---
## Punto 3 — Superposición de fuerzas (Ing. Física/Electrónica)  ★★★  [PRC, UCM]
Una carga  q₃ = 1,00 µC  está en el punto medio entre  q₁ = +2,00 µC  y  q₂ = −2,00 µC, separadas
0,300 m (0,150 m a cada lado). Se busca la fuerza neta sobre q₃.

**Principio de superposición:**  la fuerza neta es la suma vectorial de las fuerzas individuales.

> 🔧 **Aplicación en ingeniería:** en un sistema de varias cargas, la fuerza sobre una de ellas es la
> suma de todas las contribuciones. Este principio permite analizar configuraciones complejas
> (electrodos, sensores, distribuciones de carga) sumando efectos individuales.

> 🤖 **TIP-IA — Verificador crítico:**
> Escribe: *"Calculé la fuerza neta sobre una carga sumando las fuerzas de otras dos. Actúa como evaluador escéptico: pregúntame si sumé como VECTORES (con dirección) o solo las magnitudes, y si consideré bien los signos de atracción/repulsión. No me des la solución."* 

In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

# Registro global para la calificación formativa ponderada
try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))


def _num_coma(s):
    import re as _re
    # reemplaza punto decimal por coma en números dentro de un string ya formateado
    return _re.sub(r"(\d)\.(\d)", r"\1,\2", s)
def print_es(*args, **kwargs):
    args2=[_num_coma(a) if isinstance(a,str) else a for a in args]
    print(*args2, **kwargs)

def mostrar_solucion(func_solucion):
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output(); func_solucion()
    boton.on_click(_click)
    display(boton, salida)

try:
    print_es
except NameError:
    def print_es(*args, **kwargs):
        import re as _re
        print(*[_re.sub(r"(\d)\.(\d)", r"\1,\2", a) if isinstance(a, str) else a for a in args], **kwargs)
# --- Cuestionarios Física BR v3: opciones barajadas, crédito por intento, nota sobre el total ---
import random as _rnd_br
try:
    PESOS_BR
except NameError:
    PESOS_BR = {}      # id_preg -> peso de cada pregunta mostrada
try:
    INTENTOS_BR
except NameError:
    INTENTOS_BR = {}   # id_preg -> [intentos, resuelta]
_CREDITO_BR = {1: 1.0, 2: 0.5, 3: 0.25}   # crédito según el intento en que se acierta
_MAX_INT_BR = 3

def _evaluar_intento(id_preg, ok, peso, explicacion, msg_ok, msg_mal):
    est = INTENTOS_BR.setdefault(id_preg, [0, False])
    if est[1]:
        print("✔ Ya resolviste esta pregunta; tu crédito quedó registrado.")
        if explicacion: print_es("→ " + explicacion)
        return
    if est[0] >= _MAX_INT_BR:
        print("⛔ Agotaste los 3 intentos con crédito: esta pregunta ya no suma.")
        if explicacion: print_es("→ " + explicacion)
        return
    est[0] += 1
    n = est[0]
    if ok:
        est[1] = True
        cred = _CREDITO_BR.get(n, 0.0)
        REGISTRO_BR[id_preg] = (cred, float(peso))
        print_es(msg_ok + f"  (intento {n}: crédito {int(cred*100)} %)")
        if explicacion: print_es("→ " + explicacion)
    else:
        REGISTRO_BR.setdefault(id_preg, (0.0, float(peso)))
        if n < _MAX_INT_BR:
            print(msg_mal + f"  Intento {n} de {_MAX_INT_BR}. Revisa tu razonamiento antes de volver a intentar.")
        else:
            print(msg_mal + " Agotaste los 3 intentos: esta pregunta ya no suma. Estudia la explicación:")
            if explicacion: print_es("→ " + explicacion)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    orden = list(range(len(opciones))); _rnd_br.shuffle(orden)   # orden distinto en cada sesión
    correcta = orden.index(indice_correcto)
    letras = "abcdefgh"
    etiquetas = [f"{letras[i]}) {opciones[j]}" for i, j in enumerate(orden)]
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=etiquetas, value=None, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            if radio.index is None:
                print("⚠ Elige una opción."); return
            _evaluar_intento(id_preg, radio.index == correcta, peso, explicacion,
                             "✅ ¡Correcto!", "❌ Incorrecto.")
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.strip().replace(",", "."))
            except ValueError: print("⚠ Escribe un número (usa coma o punto decimal)."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            _evaluar_intento(id_preg, ok, peso, explicacion,
                             f"✅ ¡Correcto!  ({valor_correcto:.3g} {unidad})", "❌ Fuera de rango.")
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not PESOS_BR:
                print("No se encontraron cuestionarios en este taller."); return
            peso_total = sum(PESOS_BR.values())
            peso_ok = sum(float(c)*p for c, p in REGISTRO_BR.values())
            nota = 5.0*peso_ok/peso_total if peso_total > 0 else 0.0
            total = len(PESOS_BR)
            resp = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0])[0] > 0)
            prim = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0, False])[1] and INTENTOS_BR[k][0] == 1)
            print("="*46)
            print("          CALIFICACIÓN FORMATIVA")
            print("="*46)
            print(f"  Preguntas respondidas:       {resp} de {total}")
            print(f"  Acertadas al primer intento: {prim}")
            print_es(f"  Puntaje ponderado:           {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*46)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*46)
            if resp < total: print(f"  ⚠ Faltan {total-resp} preguntas: cuentan como 0.")
            if nota >= 3.5: print("  ✅ ¡Buen trabajo! Dominas los conceptos.")
            elif nota >= 3.0: print("  🟡 Vas bien, repasa lo que fallaste.")
            else: print("  🔴 Repasa la teoría y vuelve a los puntos.")
            print("\n  Crédito por pregunta: 100 % al primer intento,")
            print("  50 % al segundo, 25 % al tercero. Predice y")
            print("  razona ANTES de verificar: adivinar cuesta nota.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>📊 Tu calificación del taller</h3>"
        "<i>La nota se calcula sobre TODAS las preguntas del taller y premia acertar al primer intento.</i>"),
        boton, salida]))


k = 8.99e9
e = 1.60e-19

def esquema():
    fig,ax=plt.subplots(figsize=(6.5,5))
    # carga central (donde calculamos la fuerza) y dos cargas fuente
    q0=(2,1); q1=(0,3); q2=(4,3)
    ax.scatter([q0[0]],[q0[1]],s=300,color='#f39c12',zorder=5,edgecolor='white',linewidth=1.5)
    ax.text(q0[0],q0[1],'q₀',ha='center',va='center',color='white',fontsize=11,fontweight='bold')
    ax.scatter([q1[0]],[q1[1]],s=250,color='#c0392b',zorder=5,edgecolor='white',linewidth=1.5)
    ax.text(q1[0],q1[1],'+',ha='center',va='center',color='white',fontsize=14,fontweight='bold')
    ax.scatter([q2[0]],[q2[1]],s=250,color='#c0392b',zorder=5,edgecolor='white',linewidth=1.5)
    ax.text(q2[0],q2[1],'+',ha='center',va='center',color='white',fontsize=14,fontweight='bold')
    # fuerzas individuales (repulsión, alejándose de cada fuente)
    import numpy as _np
    def vec(desde,hasta,longitud=1.3):
        d=_np.array(hasta)-_np.array(desde); d=d/_np.linalg.norm(d)*longitud; return d
    f1=vec(q1,q0); f2=vec(q2,q0)
    ax.annotate('',xy=(q0[0]+f1[0],q0[1]+f1[1]),xytext=q0,arrowprops=dict(arrowstyle='-|>',color='#e74c3c',lw=2))
    ax.text(q0[0]+f1[0]-0.3,q0[1]+f1[1]-0.3,'F₁',color='#e74c3c',fontsize=10,fontweight='bold')
    ax.annotate('',xy=(q0[0]+f2[0],q0[1]+f2[1]),xytext=q0,arrowprops=dict(arrowstyle='-|>',color='#2980b9',lw=2))
    ax.text(q0[0]+f2[0]+0.1,q0[1]+f2[1]-0.3,'F₂',color='#2980b9',fontsize=10,fontweight='bold')
    # resultante (suma)
    fr=f1+f2
    ax.annotate('',xy=(q0[0]+fr[0],q0[1]+fr[1]),xytext=q0,arrowprops=dict(arrowstyle='-|>',color='#8e44ad',lw=3))
    ax.text(q0[0]+fr[0],q0[1]+fr[1]-0.4,'F neta',color='#8e44ad',fontsize=11,fontweight='bold',ha='center')
    ax.text(2,-0.3,'La fuerza total es la SUMA VECTORIAL de las fuerzas individuales',
            ha='center',fontsize=9,color='#2c3e50',fontweight='bold')
    ax.set_xlim(-1,5); ax.set_ylim(-0.7,4); ax.axis('off'); ax.set_aspect('equal')
    ax.set_title('Superposición de fuerzas')
    plt.tight_layout(); plt.show()
esquema()

def superposicion(q3_uC=1.0, r_lado=0.15):
    q1=2e-6; q2=2e-6; q3=q3_uC*1e-6
    F13=k*q1*q3/r_lado**2; F23=k*q2*q3/r_lado**2; Fneta=F13+F23
    rs=np.linspace(0.05,0.3,100); Fnetas=2*k*q1*q3/rs**2
    plt.figure(figsize=(9,4)); plt.plot(rs,Fnetas,lw=2,color='purple')
    plt.scatter([r_lado],[Fneta],color='red',s=60,zorder=5,label=f'r={r_lado:.2f} m → F={Fneta:.2f} N')
    plt.xlabel('distancia a cada carga (m)'); plt.ylabel('fuerza neta (N)')
    plt.title('Ambas fuerzas apuntan igual → se SUMAN'); plt.legend(); plt.grid(alpha=.3)
    plt.tight_layout(); plt.show()

interact(superposicion,
    q3_uC=FloatSlider(value=1.0,min=0.5,max=3,step=0.5,description='q3 (µC)'),
    r_lado=FloatSlider(value=0.15,min=0.05,max=0.3,step=0.05,description='r lado (m)'));

def solucion():
    import re as _re
    def _pc(*a,**k):
        def _f(t):
            if not isinstance(t,str): return t
            _sup='⁰¹²³⁴⁵⁶⁷⁸⁹'
            def _sci(m):
                mant=m.group(1); exp=int(m.group(2))
                e=('⁻' if exp<0 else '')+''.join(_sup[int(d)] for d in str(abs(exp)))
                return mant+'×10'+e
            t=_re.sub(r'(\d+(?:\.\d+)?)[eE]([+-]?\d+)', _sci, t)
            _u=r'(?:s|km/s|km|m/s²|m/s|m|N·m|N|J|W|A|V|T|Ω|C|F|Hz|rad/s²|rad/s|rad|kg·m²/s|kg·m²|kg|Pa|kPa|µF|nC|µC|pF|mWb|Wb|µT|mT|kW|MW|mA|kΩ|MΩ|rpm|°)'
            def _c3(m):
                num=m.group(2); n=int(num)
                if n==0: return m.group(0)
                if len(num)==1: nv=num+'.00'
                elif len(num)==2: nv=num+'.0'
                else: nv=num
                return m.group(1)+nv+m.group(3)
            t=_re.sub(r'(?<![\d.,])([\s=(×])(\d{1,3})(\s'+_u+r'\b)', _c3, t)
            t=_re.sub(r'(\d)\.(\d)', r'\1,\2', t)
            return t
        print(*[_f(x) for x in a], **k)
    q1=2e-6; q3=1e-6; r=0.15
    F=k*q1*q3/r**2
    _pc(f"Fuerza de q1 (+) sobre q3 (+): repele hacia q2 → F13 = {F:.3g} N")
    _pc(f"Fuerza de q2 (−) sobre q3 (+): atrae hacia q2 → F23 = {F:.3g} N")
    _pc(f"Ambas apuntan en el mismo sentido (hacia q2), así que se SUMAN:")
    _pc(f"F neta = F13 + F23 = {2*F:.3g} N")
mostrar_solucion(solucion)

print("\n--- Cuestionario Punto 3 ---")
quiz_numerico("La fuerza neta sobre q₃=1,00 µC en el punto medio entre +2,00 µC y −2,00 µC (0,150 m a cada lado) es:",1.60,0.05,"N","F13+F23=0,799+0,799≈1,60 N.", id_preg="Coulom_p3_6", peso=3.0)
quiz_opcion_multiple("Las dos fuerzas sobre q₃ (de q₁ y q₂) se suman porque:",
    ['Son del mismo valor (por eso se suman)', 'Apuntan en el mismo sentido', 'Las cargas q₁ y q₂ son iguales', 'El punto es el centro del sistema'],1,
    "La repulsión de q1 y la atracción de q2 apuntan ambas hacia q2: se suman.", id_preg="Coulom_p3_7", peso=3.0)
quiz_opcion_multiple('DEFIENDE TU CONCLUSIÓN: en este punto, sumar las magnitudes F13 + F23 dio la respuesta correcta. Un estudiante concluye que la fuerza neta SIEMPRE se obtiene sumando magnitudes. ¿Correcto o solo convincente?',
    ['Solo convincente: aquí ambas fuerzas apuntan igual; en general la suma es vectorial',
     'Correcto: el principio de superposición dice que la fuerza neta es la suma de las magnitudes',
     'Correcto: si un método da bien en un caso ya verificado, es válido en general',
     'Solo convincente: hay que restar las magnitudes siempre que las dos cargas que actúan sobre q₃ tengan signos opuestos'],
    0,
    'q₃ es repelida por +q₁ y atraída por −q₂: ambas fuerzas apuntan hacia q₂ y se suman. Con dos cargas iguales a los lados se cancelarían. Superposición es suma VECTORIAL; un caso que funciona no prueba una regla general.',
    id_preg='Coul_p3_concep_3', peso=2.0)

---
## Punto 4 — Campo eléctrico de una carga puntual (Ing. Eléctrica/Electrónica)  ★★  [CL, UCM]
Una carga puntual  Q = 8,00 nC  crea un campo eléctrico a su alrededor. Se busca la magnitud del
campo a  r = 0,100 m, y la fuerza sobre una carga de prueba de 2,00 nC ubicada allí.

**Campo de carga puntual:**  E = k·|Q|/r²    **Fuerza:**  F = q·E

> 🔧 **Aplicación en ingeniería:** el campo eléctrico describe la influencia de una carga en el
> espacio, sin necesidad de otra carga presente. Es el concepto que permite diseñar y analizar
> capacitores, sensores, pantallas y sistemas de alto voltaje.

> 🤖 **TIP-IA — Traductor de representaciones:**
> Pega la relación E=kQ/r² a un asistente: *"Explícame la diferencia entre el CAMPO eléctrico en un punto y la FUERZA sobre una carga en ese punto. ¿Por qué el campo existe aunque no haya ninguna carga de prueba ahí? Dame la intuición, no la fórmula."* 

In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

# Registro global para la calificación formativa ponderada
try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))


def _num_coma(s):
    import re as _re
    # reemplaza punto decimal por coma en números dentro de un string ya formateado
    return _re.sub(r"(\d)\.(\d)", r"\1,\2", s)
def print_es(*args, **kwargs):
    args2=[_num_coma(a) if isinstance(a,str) else a for a in args]
    print(*args2, **kwargs)

def mostrar_solucion(func_solucion):
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output(); func_solucion()
    boton.on_click(_click)
    display(boton, salida)

try:
    print_es
except NameError:
    def print_es(*args, **kwargs):
        import re as _re
        print(*[_re.sub(r"(\d)\.(\d)", r"\1,\2", a) if isinstance(a, str) else a for a in args], **kwargs)
# --- Cuestionarios Física BR v3: opciones barajadas, crédito por intento, nota sobre el total ---
import random as _rnd_br
try:
    PESOS_BR
except NameError:
    PESOS_BR = {}      # id_preg -> peso de cada pregunta mostrada
try:
    INTENTOS_BR
except NameError:
    INTENTOS_BR = {}   # id_preg -> [intentos, resuelta]
_CREDITO_BR = {1: 1.0, 2: 0.5, 3: 0.25}   # crédito según el intento en que se acierta
_MAX_INT_BR = 3

def _evaluar_intento(id_preg, ok, peso, explicacion, msg_ok, msg_mal):
    est = INTENTOS_BR.setdefault(id_preg, [0, False])
    if est[1]:
        print("✔ Ya resolviste esta pregunta; tu crédito quedó registrado.")
        if explicacion: print_es("→ " + explicacion)
        return
    if est[0] >= _MAX_INT_BR:
        print("⛔ Agotaste los 3 intentos con crédito: esta pregunta ya no suma.")
        if explicacion: print_es("→ " + explicacion)
        return
    est[0] += 1
    n = est[0]
    if ok:
        est[1] = True
        cred = _CREDITO_BR.get(n, 0.0)
        REGISTRO_BR[id_preg] = (cred, float(peso))
        print_es(msg_ok + f"  (intento {n}: crédito {int(cred*100)} %)")
        if explicacion: print_es("→ " + explicacion)
    else:
        REGISTRO_BR.setdefault(id_preg, (0.0, float(peso)))
        if n < _MAX_INT_BR:
            print(msg_mal + f"  Intento {n} de {_MAX_INT_BR}. Revisa tu razonamiento antes de volver a intentar.")
        else:
            print(msg_mal + " Agotaste los 3 intentos: esta pregunta ya no suma. Estudia la explicación:")
            if explicacion: print_es("→ " + explicacion)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    orden = list(range(len(opciones))); _rnd_br.shuffle(orden)   # orden distinto en cada sesión
    correcta = orden.index(indice_correcto)
    letras = "abcdefgh"
    etiquetas = [f"{letras[i]}) {opciones[j]}" for i, j in enumerate(orden)]
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=etiquetas, value=None, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            if radio.index is None:
                print("⚠ Elige una opción."); return
            _evaluar_intento(id_preg, radio.index == correcta, peso, explicacion,
                             "✅ ¡Correcto!", "❌ Incorrecto.")
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.strip().replace(",", "."))
            except ValueError: print("⚠ Escribe un número (usa coma o punto decimal)."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            _evaluar_intento(id_preg, ok, peso, explicacion,
                             f"✅ ¡Correcto!  ({valor_correcto:.3g} {unidad})", "❌ Fuera de rango.")
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not PESOS_BR:
                print("No se encontraron cuestionarios en este taller."); return
            peso_total = sum(PESOS_BR.values())
            peso_ok = sum(float(c)*p for c, p in REGISTRO_BR.values())
            nota = 5.0*peso_ok/peso_total if peso_total > 0 else 0.0
            total = len(PESOS_BR)
            resp = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0])[0] > 0)
            prim = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0, False])[1] and INTENTOS_BR[k][0] == 1)
            print("="*46)
            print("          CALIFICACIÓN FORMATIVA")
            print("="*46)
            print(f"  Preguntas respondidas:       {resp} de {total}")
            print(f"  Acertadas al primer intento: {prim}")
            print_es(f"  Puntaje ponderado:           {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*46)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*46)
            if resp < total: print(f"  ⚠ Faltan {total-resp} preguntas: cuentan como 0.")
            if nota >= 3.5: print("  ✅ ¡Buen trabajo! Dominas los conceptos.")
            elif nota >= 3.0: print("  🟡 Vas bien, repasa lo que fallaste.")
            else: print("  🔴 Repasa la teoría y vuelve a los puntos.")
            print("\n  Crédito por pregunta: 100 % al primer intento,")
            print("  50 % al segundo, 25 % al tercero. Predice y")
            print("  razona ANTES de verificar: adivinar cuesta nota.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>📊 Tu calificación del taller</h3>"
        "<i>La nota se calcula sobre TODAS las preguntas del taller y premia acertar al primer intento.</i>"),
        boton, salida]))


k = 8.99e9
e = 1.60e-19

def esquema():
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(9,4))
    # carga positiva: campo sale; negativa: campo entra
    for ax,signo,col,titulo in [(ax1,'+','#c0392b','Carga + : campo SALE'),(ax2,'−','#2980b9','Carga − : campo ENTRA')]:
        ax.scatter([0],[0],s=400,color=col,zorder=5,edgecolor='white',linewidth=1.5)
        ax.text(0,0,signo,ha='center',va='center',color='white',fontsize=16,fontweight='bold')
        # líneas radiales
        for ang in np.linspace(0,2*np.pi,12,endpoint=False):
            x,y=np.cos(ang),np.sin(ang)
            if signo=='+':
                ax.annotate('',xy=(x*2,y*2),xytext=(x*0.5,y*0.5),arrowprops=dict(arrowstyle='-|>',color=col,lw=1.3,alpha=0.7))
            else:
                ax.annotate('',xy=(x*0.5,y*0.5),xytext=(x*2,y*2),arrowprops=dict(arrowstyle='-|>',color=col,lw=1.3,alpha=0.7))
        ax.set_xlim(-2.5,2.5); ax.set_ylim(-2.5,2.5); ax.axis('off'); ax.set_aspect('equal')
        ax.set_title(titulo,fontsize=10,color=col)
    plt.suptitle('Campo eléctrico de una carga puntual (E = k·q/r²)',fontsize=11,fontweight='bold',color='#2c3e50')
    plt.tight_layout(); plt.show()
esquema()

def campo(Q_nC=8.0, r=0.10):
    Q=Q_nC*1e-9; E=k*Q/r**2
    rs=np.linspace(0.03,0.4,100); Es=k*Q/rs**2
    plt.figure(figsize=(9,4)); plt.plot(rs,Es,lw=2,color='navy')
    plt.scatter([r],[E],color='red',s=60,zorder=5,label=f'r={r:.2f} m → E={E:.0f} N/C')
    plt.xlabel('distancia r (m)'); plt.ylabel('campo E (N/C)')
    plt.title('Campo de carga puntual (E ∝ 1/r²)'); plt.legend(); plt.grid(alpha=.3)
    plt.tight_layout(); plt.show()

interact(campo,
    Q_nC=FloatSlider(value=8.0,min=1,max=20,step=1,description='Q (nC)'),
    r=FloatSlider(value=0.10,min=0.03,max=0.4,step=0.01,description='r (m)'));

def solucion():
    import re as _re
    def _pc(*a,**k):
        def _f(t):
            if not isinstance(t,str): return t
            _sup='⁰¹²³⁴⁵⁶⁷⁸⁹'
            def _sci(m):
                mant=m.group(1); exp=int(m.group(2))
                e=('⁻' if exp<0 else '')+''.join(_sup[int(d)] for d in str(abs(exp)))
                return mant+'×10'+e
            t=_re.sub(r'(\d+(?:\.\d+)?)[eE]([+-]?\d+)', _sci, t)
            _u=r'(?:s|km/s|km|m/s²|m/s|m|N·m|N|J|W|A|V|T|Ω|C|F|Hz|rad/s²|rad/s|rad|kg·m²/s|kg·m²|kg|Pa|kPa|µF|nC|µC|pF|mWb|Wb|µT|mT|kW|MW|mA|kΩ|MΩ|rpm|°)'
            def _c3(m):
                num=m.group(2); n=int(num)
                if n==0: return m.group(0)
                if len(num)==1: nv=num+'.00'
                elif len(num)==2: nv=num+'.0'
                else: nv=num
                return m.group(1)+nv+m.group(3)
            t=_re.sub(r'(?<![\d.,])([\s=(×])(\d{1,3})(\s'+_u+r'\b)', _c3, t)
            t=_re.sub(r'(\d)\.(\d)', r'\1,\2', t)
            return t
        print(*[_f(x) for x in a], **k)
    Q=8e-9; r=0.10; E=k*Q/r**2; q=2e-9
    _pc(f"Campo: E = k·|Q|/r² = 8,99×10⁹ × 8,00×10⁻⁹/(0,100)² = {E:.3g} N/C")
    _pc(f"Fuerza sobre q=2,00 nC: F = q·E = 2,00×10⁻⁹ × {E:.0f} = {q*E:.3g} N")
    _pc("El campo existe aunque no haya carga de prueba; F=qE aparece al colocar una carga.")
mostrar_solucion(solucion)

print("\n--- Cuestionario Punto 4 ---")
quiz_numerico("El campo eléctrico de una carga de 8,00 nC a 0,100 m es:",7190,0.04,"N/C","E=kQ/r²≈7,19×10³ N/C.", id_preg="Coulom_p4_8", peso=2.0)
quiz_opcion_multiple("El campo eléctrico en un punto describe:",
    ['Solo la fuerza sobre la carga que se coloque allí, si la hay', 'La influencia de la fuente en ese punto, haya o no otra carga', 'La energía total del sistema de cargas en ese punto', 'La carga total que hay en esa región del espacio'],1,
    "El campo existe por la carga fuente; F=qE aparece al poner una carga de prueba.", id_preg="Coulom_p4_9", peso=2.0)
quiz_opcion_multiple("DEFIENDE TU CONCLUSIÓN: un estudiante dice: 'si no hay una carga de prueba en un punto, ahí no hay campo eléctrico'. ¿Correcto o solo convincente?",
    ['Solo convincente: el campo lo crea la carga fuente; la de prueba solo lo revela',
     'Correcto: el campo es la fuerza sobre una carga, y sin carga no hay fuerza ni campo',
     'Correcto: el campo aparece en el punto cuando se coloca allí la carga de prueba, porque es ella misma la que lo genera',
     'Solo convincente: hay campo, pero solo donde haya materia que pueda polarizarse'],
    0,
    'E = F/q₀ es una forma de MEDIR el campo, no la causa de que exista: E = kQ/r² no depende de q₀. El campo existe también en el vacío (la luz es un campo que viaja sin materia).',
    id_preg='Coul_p4_concep_4', peso=2.0)

---
## Punto 5 — Campo eléctrico neto de dos cargas (Ing. Física/Instrumentación)  ★★★  [PRC, ULC]
Dos cargas iguales de  +4,00 nC  están separadas  0,100 m. Se busca el campo eléctrico neto en el
punto medio entre ellas.

**Superposición del campo:**  el campo neto es la suma vectorial de los campos de cada carga.

> 🔧 **Aplicación en ingeniería:** el campo neto de varias cargas se obtiene sumando vectorialmente.
> En el punto medio entre dos cargas iguales, los campos se cancelan (punto de campo nulo), un
> concepto usado en el diseño de configuraciones de electrodos y en instrumentación de precisión.

> 🤖 **TIP-IA — Pon a prueba tu intuición:**
> Pídele: *"Voy a sostener que en el punto medio entre dos cargas positivas iguales el campo eléctrico es máximo. Dame el mejor contraargumento y dime qué pasa realmente ahí."* (Pista: piensa en la simetría y en cómo se suman los vectores). 

In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

# Registro global para la calificación formativa ponderada
try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))


def _num_coma(s):
    import re as _re
    # reemplaza punto decimal por coma en números dentro de un string ya formateado
    return _re.sub(r"(\d)\.(\d)", r"\1,\2", s)
def print_es(*args, **kwargs):
    args2=[_num_coma(a) if isinstance(a,str) else a for a in args]
    print(*args2, **kwargs)

def mostrar_solucion(func_solucion):
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output(); func_solucion()
    boton.on_click(_click)
    display(boton, salida)

try:
    print_es
except NameError:
    def print_es(*args, **kwargs):
        import re as _re
        print(*[_re.sub(r"(\d)\.(\d)", r"\1,\2", a) if isinstance(a, str) else a for a in args], **kwargs)
# --- Cuestionarios Física BR v3: opciones barajadas, crédito por intento, nota sobre el total ---
import random as _rnd_br
try:
    PESOS_BR
except NameError:
    PESOS_BR = {}      # id_preg -> peso de cada pregunta mostrada
try:
    INTENTOS_BR
except NameError:
    INTENTOS_BR = {}   # id_preg -> [intentos, resuelta]
_CREDITO_BR = {1: 1.0, 2: 0.5, 3: 0.25}   # crédito según el intento en que se acierta
_MAX_INT_BR = 3

def _evaluar_intento(id_preg, ok, peso, explicacion, msg_ok, msg_mal):
    est = INTENTOS_BR.setdefault(id_preg, [0, False])
    if est[1]:
        print("✔ Ya resolviste esta pregunta; tu crédito quedó registrado.")
        if explicacion: print_es("→ " + explicacion)
        return
    if est[0] >= _MAX_INT_BR:
        print("⛔ Agotaste los 3 intentos con crédito: esta pregunta ya no suma.")
        if explicacion: print_es("→ " + explicacion)
        return
    est[0] += 1
    n = est[0]
    if ok:
        est[1] = True
        cred = _CREDITO_BR.get(n, 0.0)
        REGISTRO_BR[id_preg] = (cred, float(peso))
        print_es(msg_ok + f"  (intento {n}: crédito {int(cred*100)} %)")
        if explicacion: print_es("→ " + explicacion)
    else:
        REGISTRO_BR.setdefault(id_preg, (0.0, float(peso)))
        if n < _MAX_INT_BR:
            print(msg_mal + f"  Intento {n} de {_MAX_INT_BR}. Revisa tu razonamiento antes de volver a intentar.")
        else:
            print(msg_mal + " Agotaste los 3 intentos: esta pregunta ya no suma. Estudia la explicación:")
            if explicacion: print_es("→ " + explicacion)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    orden = list(range(len(opciones))); _rnd_br.shuffle(orden)   # orden distinto en cada sesión
    correcta = orden.index(indice_correcto)
    letras = "abcdefgh"
    etiquetas = [f"{letras[i]}) {opciones[j]}" for i, j in enumerate(orden)]
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=etiquetas, value=None, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            if radio.index is None:
                print("⚠ Elige una opción."); return
            _evaluar_intento(id_preg, radio.index == correcta, peso, explicacion,
                             "✅ ¡Correcto!", "❌ Incorrecto.")
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.strip().replace(",", "."))
            except ValueError: print("⚠ Escribe un número (usa coma o punto decimal)."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            _evaluar_intento(id_preg, ok, peso, explicacion,
                             f"✅ ¡Correcto!  ({valor_correcto:.3g} {unidad})", "❌ Fuera de rango.")
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not PESOS_BR:
                print("No se encontraron cuestionarios en este taller."); return
            peso_total = sum(PESOS_BR.values())
            peso_ok = sum(float(c)*p for c, p in REGISTRO_BR.values())
            nota = 5.0*peso_ok/peso_total if peso_total > 0 else 0.0
            total = len(PESOS_BR)
            resp = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0])[0] > 0)
            prim = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0, False])[1] and INTENTOS_BR[k][0] == 1)
            print("="*46)
            print("          CALIFICACIÓN FORMATIVA")
            print("="*46)
            print(f"  Preguntas respondidas:       {resp} de {total}")
            print(f"  Acertadas al primer intento: {prim}")
            print_es(f"  Puntaje ponderado:           {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*46)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*46)
            if resp < total: print(f"  ⚠ Faltan {total-resp} preguntas: cuentan como 0.")
            if nota >= 3.5: print("  ✅ ¡Buen trabajo! Dominas los conceptos.")
            elif nota >= 3.0: print("  🟡 Vas bien, repasa lo que fallaste.")
            else: print("  🔴 Repasa la teoría y vuelve a los puntos.")
            print("\n  Crédito por pregunta: 100 % al primer intento,")
            print("  50 % al segundo, 25 % al tercero. Predice y")
            print("  razona ANTES de verificar: adivinar cuesta nota.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>📊 Tu calificación del taller</h3>"
        "<i>La nota se calcula sobre TODAS las preguntas del taller y premia acertar al primer intento.</i>"),
        boton, salida]))


k = 8.99e9
e = 1.60e-19

def esquema():
    fig,ax=plt.subplots(figsize=(7,4.5))
    q1=(0,0); q2=(4,0); P=(2,2)
    ax.scatter([q1[0]],[q1[1]],s=280,color='#c0392b',zorder=5,edgecolor='white',linewidth=1.5)
    ax.text(q1[0],q1[1],'+',ha='center',va='center',color='white',fontsize=14,fontweight='bold')
    ax.scatter([q2[0]],[q2[1]],s=280,color='#c0392b',zorder=5,edgecolor='white',linewidth=1.5)
    ax.text(q2[0],q2[1],'+',ha='center',va='center',color='white',fontsize=14,fontweight='bold')
    ax.scatter([P[0]],[P[1]],s=80,color='#2c3e50',zorder=6)
    ax.text(P[0],P[1]+0.3,'P',ha='center',fontsize=10,color='#2c3e50',fontweight='bold')
    import numpy as _np
    def vec(desde,hasta,L=1.3):
        d=_np.array(hasta)-_np.array(desde); return d/_np.linalg.norm(d)*L
    e1=vec(q1,P); e2=vec(q2,P)
    ax.annotate('',xy=(P[0]+e1[0],P[1]+e1[1]),xytext=P,arrowprops=dict(arrowstyle='-|>',color='#e74c3c',lw=2))
    ax.text(P[0]+e1[0]+0.1,P[1]+e1[1],'E₁',color='#e74c3c',fontsize=10,fontweight='bold')
    ax.annotate('',xy=(P[0]+e2[0],P[1]+e2[1]),xytext=P,arrowprops=dict(arrowstyle='-|>',color='#2980b9',lw=2))
    ax.text(P[0]+e2[0]-0.4,P[1]+e2[1],'E₂',color='#2980b9',fontsize=10,fontweight='bold')
    er=e1+e2
    ax.annotate('',xy=(P[0]+er[0],P[1]+er[1]),xytext=P,arrowprops=dict(arrowstyle='-|>',color='#8e44ad',lw=3))
    ax.text(P[0]+er[0],P[1]+er[1]+0.2,'E neto',color='#8e44ad',fontsize=11,fontweight='bold',ha='center')
    ax.text(2,-0.8,'El campo neto es la SUMA VECTORIAL de los campos de cada carga',
            ha='center',fontsize=9,color='#2c3e50',fontweight='bold')
    ax.set_xlim(-1,5); ax.set_ylim(-1.2,4); ax.axis('off'); ax.set_aspect('equal')
    ax.set_title('Campo eléctrico neto de dos cargas')
    plt.tight_layout(); plt.show()
esquema()

def campo_neto(Q_nC=4.0, sep=0.10):
    Q=Q_nC*1e-9; r=sep/2
    E_cada=k*Q/r**2; E_neto=0.0  # se cancelan en el medio (cargas iguales)
    # mostrar el campo de cada una y el neto a lo largo del eje
    xs=np.linspace(0.01,sep-0.01,100)
    E1=k*Q/xs**2; E2=k*Q/(sep-xs)**2
    Eneto=E1-E2  # signo: apuntan en sentidos opuestos hacia el centro
    plt.figure(figsize=(9,4))
    plt.plot(xs*100,Eneto,lw=2,color='purple',label='campo neto')
    plt.axhline(0,color='k',lw=.5); plt.axvline(sep/2*100,color='green',ls='--',alpha=.6,label='punto medio (E=0)')
    plt.xlabel('posición sobre el eje (cm)'); plt.ylabel('campo neto (N/C)')
    plt.title('El campo neto se anula en el punto medio'); plt.legend(); plt.grid(alpha=.3)
    plt.tight_layout(); plt.show()
    print(f"Campo de cada carga a {r:.3f} m del centro: {E_cada:.3g} N/C (se cancelan en el medio)".replace(".",","))

interact(campo_neto,
    Q_nC=FloatSlider(value=4.0,min=1,max=10,step=1,description='Q (nC)'),
    sep=FloatSlider(value=0.10,min=0.04,max=0.3,step=0.02,description='separación (m)'));

def solucion():
    import re as _re
    def _pc(*a,**k):
        def _f(t):
            if not isinstance(t,str): return t
            _sup='⁰¹²³⁴⁵⁶⁷⁸⁹'
            def _sci(m):
                mant=m.group(1); exp=int(m.group(2))
                e=('⁻' if exp<0 else '')+''.join(_sup[int(d)] for d in str(abs(exp)))
                return mant+'×10'+e
            t=_re.sub(r'(\d+(?:\.\d+)?)[eE]([+-]?\d+)', _sci, t)
            _u=r'(?:s|km/s|km|m/s²|m/s|m|N·m|N|J|W|A|V|T|Ω|C|F|Hz|rad/s²|rad/s|rad|kg·m²/s|kg·m²|kg|Pa|kPa|µF|nC|µC|pF|mWb|Wb|µT|mT|kW|MW|mA|kΩ|MΩ|rpm|°)'
            def _c3(m):
                num=m.group(2); n=int(num)
                if n==0: return m.group(0)
                if len(num)==1: nv=num+'.00'
                elif len(num)==2: nv=num+'.0'
                else: nv=num
                return m.group(1)+nv+m.group(3)
            t=_re.sub(r'(?<![\d.,])([\s=(×])(\d{1,3})(\s'+_u+r'\b)', _c3, t)
            t=_re.sub(r'(\d)\.(\d)', r'\1,\2', t)
            return t
        print(*[_f(x) for x in a], **k)
    Q=4e-9; r=0.05
    E=k*Q/r**2
    _pc(f"Cada carga produce en el punto medio: E = k·Q/r² = 8,99×10⁹×4,00×10⁻⁹/(0,0500)² = {E:.3g} N/C")
    _pc(f"Pero apuntan en SENTIDOS OPUESTOS (ambas cargas son +):")
    _pc(f"E neto = {E:.0f} − {E:.0f} = 0 N/C")
    _pc("En el punto medio entre dos cargas IGUALES, el campo neto es cero.")
mostrar_solucion(solucion)

print("\n--- Cuestionario Punto 5 ---")
quiz_numerico("El campo que produce UNA de las cargas (+4,00 nC) en el punto medio, a 0,0500 m, es (en N/C):",14400,0.05,"N/C","E=kQ/r²≈1,44×10⁴ N/C.", id_preg="Coulom_p5_10", peso=3.0)
quiz_opcion_multiple("En el punto medio entre dos cargas positivas iguales, el campo eléctrico neto es:",
    ['Máximo (está cerca de ambas)', 'Cero (los campos se cancelan)', 'El doble del de una carga', 'Negativo (apunta hacia adentro)'],1,
    "Los campos de ambas apuntan en sentidos opuestos y se cancelan: E neto = 0.", id_preg="Coulom_p5_11", peso=3.0)
quiz_opcion_multiple('DEFIENDE TU CONCLUSIÓN: en el punto medio entre dos cargas positivas iguales, un estudiante dice que el campo es máximo por estar cerca de ambas. ¿Correcto o solo convincente?',
    ['Solo convincente: los dos campos tienen igual magnitud y sentidos opuestos; el campo neto es cero',
     'Correcto: al estar cerca de ambas cargas, los dos campos se suman y el resultado es máximo',
     'Correcto: el campo es máximo ahí, porque el potencial también es máximo en ese punto',
     'Solo convincente: el campo neto es el doble del de una sola carga, pero no es el máximo'],
    0,
    'E = 0 por simetría. El potencial no es cero (ambas cargas aportan positivo) ni simplemente máximo: es mínimo a lo largo de la recta que une las cargas y máximo sobre la mediatriz (un punto de silla).',
    id_preg='Coul_p5_concep_5', peso=2.0)

---
## ✅ Fin del taller de Carga, Coulomb y Campo Eléctrico (Física BR)
Estas bases —carga, fuerza de Coulomb y campo eléctrico— abren el curso y sustentan los temas de
Gauss, potencial y capacitancia. Experimenta con los sliders y **predice** antes de pulsar
"Mostrar solución".
*Física interactiva para ingeniería — 2026-2.*

In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

# Registro global para la calificación formativa ponderada
try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))


def _num_coma(s):
    import re as _re
    # reemplaza punto decimal por coma en números dentro de un string ya formateado
    return _re.sub(r"(\d)\.(\d)", r"\1,\2", s)
def print_es(*args, **kwargs):
    args2=[_num_coma(a) if isinstance(a,str) else a for a in args]
    print(*args2, **kwargs)

def mostrar_solucion(func_solucion):
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output(); func_solucion()
    boton.on_click(_click)
    display(boton, salida)

try:
    print_es
except NameError:
    def print_es(*args, **kwargs):
        import re as _re
        print(*[_re.sub(r"(\d)\.(\d)", r"\1,\2", a) if isinstance(a, str) else a for a in args], **kwargs)
# --- Cuestionarios Física BR v3: opciones barajadas, crédito por intento, nota sobre el total ---
import random as _rnd_br
try:
    PESOS_BR
except NameError:
    PESOS_BR = {}      # id_preg -> peso de cada pregunta mostrada
try:
    INTENTOS_BR
except NameError:
    INTENTOS_BR = {}   # id_preg -> [intentos, resuelta]
_CREDITO_BR = {1: 1.0, 2: 0.5, 3: 0.25}   # crédito según el intento en que se acierta
_MAX_INT_BR = 3

def _evaluar_intento(id_preg, ok, peso, explicacion, msg_ok, msg_mal):
    est = INTENTOS_BR.setdefault(id_preg, [0, False])
    if est[1]:
        print("✔ Ya resolviste esta pregunta; tu crédito quedó registrado.")
        if explicacion: print_es("→ " + explicacion)
        return
    if est[0] >= _MAX_INT_BR:
        print("⛔ Agotaste los 3 intentos con crédito: esta pregunta ya no suma.")
        if explicacion: print_es("→ " + explicacion)
        return
    est[0] += 1
    n = est[0]
    if ok:
        est[1] = True
        cred = _CREDITO_BR.get(n, 0.0)
        REGISTRO_BR[id_preg] = (cred, float(peso))
        print_es(msg_ok + f"  (intento {n}: crédito {int(cred*100)} %)")
        if explicacion: print_es("→ " + explicacion)
    else:
        REGISTRO_BR.setdefault(id_preg, (0.0, float(peso)))
        if n < _MAX_INT_BR:
            print(msg_mal + f"  Intento {n} de {_MAX_INT_BR}. Revisa tu razonamiento antes de volver a intentar.")
        else:
            print(msg_mal + " Agotaste los 3 intentos: esta pregunta ya no suma. Estudia la explicación:")
            if explicacion: print_es("→ " + explicacion)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    orden = list(range(len(opciones))); _rnd_br.shuffle(orden)   # orden distinto en cada sesión
    correcta = orden.index(indice_correcto)
    letras = "abcdefgh"
    etiquetas = [f"{letras[i]}) {opciones[j]}" for i, j in enumerate(orden)]
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=etiquetas, value=None, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            if radio.index is None:
                print("⚠ Elige una opción."); return
            _evaluar_intento(id_preg, radio.index == correcta, peso, explicacion,
                             "✅ ¡Correcto!", "❌ Incorrecto.")
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.strip().replace(",", "."))
            except ValueError: print("⚠ Escribe un número (usa coma o punto decimal)."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            _evaluar_intento(id_preg, ok, peso, explicacion,
                             f"✅ ¡Correcto!  ({valor_correcto:.3g} {unidad})", "❌ Fuera de rango.")
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not PESOS_BR:
                print("No se encontraron cuestionarios en este taller."); return
            peso_total = sum(PESOS_BR.values())
            peso_ok = sum(float(c)*p for c, p in REGISTRO_BR.values())
            nota = 5.0*peso_ok/peso_total if peso_total > 0 else 0.0
            total = len(PESOS_BR)
            resp = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0])[0] > 0)
            prim = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0, False])[1] and INTENTOS_BR[k][0] == 1)
            print("="*46)
            print("          CALIFICACIÓN FORMATIVA")
            print("="*46)
            print(f"  Preguntas respondidas:       {resp} de {total}")
            print(f"  Acertadas al primer intento: {prim}")
            print_es(f"  Puntaje ponderado:           {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*46)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*46)
            if resp < total: print(f"  ⚠ Faltan {total-resp} preguntas: cuentan como 0.")
            if nota >= 3.5: print("  ✅ ¡Buen trabajo! Dominas los conceptos.")
            elif nota >= 3.0: print("  🟡 Vas bien, repasa lo que fallaste.")
            else: print("  🔴 Repasa la teoría y vuelve a los puntos.")
            print("\n  Crédito por pregunta: 100 % al primer intento,")
            print("  50 % al segundo, 25 % al tercero. Predice y")
            print("  razona ANTES de verificar: adivinar cuesta nota.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>📊 Tu calificación del taller</h3>"
        "<i>La nota se calcula sobre TODAS las preguntas del taller y premia acertar al primer intento.</i>"),
        boton, salida]))


# Calificación formativa del taller
calificacion_final(total_preguntas=16)
